# Qualité des synonymes ANS — investigation (chantier 2)

Suite du récap `docs/sessions/2026-05-30_refonte_dagger_asterisk.md` section 9, chantier 2 (synonymes ANS de qualité douteuse : D21.6 « Tronc », M01.08 noms anatomiques bruts).

Sources chargées via `load_exploration_context` (cf CLAUDE.md — pas de `pl.read_csv` direct dans les notebooks).

In [1]:
from __future__ import annotations

import polars as pl

from recode_icd._normalize import normalize_for_match
from recode_icd.utils.loaders_dev import load_exploration_context

ctx = load_exploration_context()
df = ctx.flat
assert df is not None, "CSV maître absent — lancer `recode-icd build flat-csv` d'abord."

# Configuration polars pour l'exploration interactive.
pl.Config.set_fmt_str_lengths(500)
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_width_chars(200)

print(f"CSV total : {df.height} lignes\n")

CSV total : 199970 lignes



## Question 1 — Volume global des synonymes ANS

In [6]:
print("=" * 70)
print("QUESTION 1 — Volume des synonymes ANS")
print("=" * 70)

syn_total = df.filter(pl.col("type") == "synonyme").height
syn_ans = df.filter((pl.col("type") == "synonyme") & (pl.col("source") == "ANS")).height
print(f"Synonymes total : {syn_total}")
print(f"Synonymes ANS : {syn_ans} ({100*syn_ans/syn_total:.1f}%)")
print("\nRépartition de TOUS les synonymes par source :")
print(
    df.filter(pl.col("type") == "synonyme")
      .group_by("source")
      .agg(pl.len().alias("n"))
      .sort("n", descending=True)
)

QUESTION 1 — Volume des synonymes ANS
Synonymes total : 64504
Synonymes ANS : 15567 (24.1%)

Répartition de TOUS les synonymes par source :
shape: (13, 2)
┌─────────────────────────────┬───────┐
│ source                      ┆ n     │
│ ---                         ┆ ---   │
│ str                         ┆ u32   │
╞═════════════════════════════╪═══════╡
│ CIM-10 index                ┆ 36627 │
│ ANS                         ┆ 15567 │
│ CIM-10                      ┆ 7123  │
│ AP-HP Dermatologie          ┆ 1551  │
│ ORPHANET                    ┆ 1334  │
│ AP-HP Rhumatologie          ┆ 617   │
│ AP-HP Néphrologie           ┆ 565   │
│ AP-HP Ophtalmologie         ┆ 281   │
│ AP-HP Endocrinologie        ┆ 263   │
│ AP-HP Troubles métaboliques ┆ 221   │
│ AP-HP Germes (SPILF)        ┆ 193   │
│ AP-HP GRONES                ┆ 117   │
│ AP-HP SRLF                  ┆ 45    │
└─────────────────────────────┴───────┘


## Question 2 — Redondance synonymes ANS vs descripteurs OFS

Pour chaque ligne `(type=synonyme, source=ANS)`, on vérifie si le couple `(code, texte normalisé)` existe aussi dans les synonymes OFS au niveau code (`source=CIM-10`, `source_level=code`).

La normalisation utilisée est `normalize_for_match` (NFKD + lowercase + ponctuation + whitespace).

In [7]:
print("=" * 70)
print("QUESTION 2 — Redondance synonymes ANS vs descripteurs OFS")
print("=" * 70)


def add_normalized_text(df_in: pl.DataFrame) -> pl.DataFrame:
    """Ajoute une colonne `texte_norm` issue de `normalize_for_match`."""
    return df_in.with_columns(
        pl.col("texte")
          .map_elements(normalize_for_match, return_dtype=pl.Utf8)
          .alias("texte_norm")
    )


syn_ans_df = add_normalized_text(
    df.filter((pl.col("type") == "synonyme") & (pl.col("source") == "ANS"))
)
syn_ofs_df = add_normalized_text(
    df.filter(
        (pl.col("type") == "synonyme")
        & (pl.col("source") == "CIM-10")
        & (pl.col("source_level") == "code")
    )
)

# Pour chaque ligne ANS, voir si le couple (code, texte_norm) existe en OFS.
joined = syn_ans_df.join(
    syn_ofs_df.select(["code", "texte_norm"])
              .with_columns(pl.lit(True).alias("has_ofs_match")),
    on=["code", "texte_norm"],
    how="left",
)
matched = joined.filter(pl.col("has_ofs_match")).height
not_matched = joined.filter(pl.col("has_ofs_match").is_null()).height
print(f"Synonymes ANS total : {syn_ans_df.height}")
print(
    f"  Dont redondants avec un descripteur OFS du même code : "
    f"{matched} ({100*matched/syn_ans_df.height:.1f}%)"
)
print(
    f"  Dont uniques (pas d'équivalent OFS) : "
    f"{not_matched} ({100*not_matched/syn_ans_df.height:.1f}%)"
)

QUESTION 2 — Redondance synonymes ANS vs descripteurs OFS
Synonymes ANS total : 15567
  Dont redondants avec un descripteur OFS du même code : 0 (0.0%)
  Dont uniques (pas d'équivalent OFS) : 15567 (100.0%)


## Question 3 — Codes dont la section « Périmètre clinique » deviendrait vide

Si on retire les synonymes ANS, combien de codes perdent toute leur section « Périmètre clinique » du prototype de fiche ?

Sources qui alimentent la section 2 du prototype :
- synonyme OFS niveau code
- inclusion OFS niveau code
- fallback ANS si vide

In [8]:
print("=" * 70)
print("QUESTION 3 — Codes dont la section Périmètre serait vide si on retire les synonymes ANS")
print("=" * 70)

# Codes qui ont au moins UN descripteur OFS ou inclusion OFS niveau code.
codes_with_ofs_perimeter = df.filter(
    ((pl.col("type") == "synonyme") | (pl.col("type") == "inclusion"))
    & (pl.col("source") == "CIM-10")
    & (pl.col("source_level") == "code")
).select("code").unique()

# Codes qui ont au moins UN synonyme ANS (= peuvent activer le fallback).
codes_with_ans_syn = df.filter(
    (pl.col("type") == "synonyme") & (pl.col("source") == "ANS")
).select("code").unique()

# Codes qui dépendent EXCLUSIVEMENT du fallback ANS pour la section 2.
codes_only_ans = codes_with_ans_syn.join(
    codes_with_ofs_perimeter, on="code", how="anti"
)
print(f"\nCodes avec section Périmètre alimentée par OFS : {codes_with_ofs_perimeter.height}")
print(f"Codes avec synonymes ANS : {codes_with_ans_syn.height}")
print(
    f"Codes qui PERDRAIENT leur section Périmètre si on retire ANS : "
    f"{codes_only_ans.height}"
)

# Échantillon de ces codes.
print("\n20 exemples de codes qui perdraient leur section Périmètre :")
sample = codes_only_ans.head(20)
for row in sample.iter_rows(named=True):
    code = row["code"]
    syn = df.filter(
        (pl.col("code") == code)
        & (pl.col("type") == "synonyme")
        & (pl.col("source") == "ANS")
    )
    libelle = syn["libelle"].first() if syn.height > 0 else "?"
    syn_texts = [r["texte"] for r in syn.iter_rows(named=True)][:3]
    print(f"  {code} ({libelle[:50]}) : {syn_texts}")

QUESTION 3 — Codes dont la section Périmètre serait vide si on retire les synonymes ANS

Codes avec section Périmètre alimentée par OFS : 3511
Codes avec synonymes ANS : 4531
Codes qui PERDRAIENT leur section Périmètre si on retire ANS : 3602

20 exemples de codes qui perdraient leur section Périmètre :
  M22.33 (Autres déplacements de la rotule - " Avant-bras ") : ['articulation du poignet', 'cubitus', 'radius']
  M94.35 (Chondrolyse - " Région pelvienne et cuisse ") : ['articulation de la hanche', 'articulation sacro-iliaque', 'bassin']
  M06.26 (Bursite rhumatoïde - " Jambe ") : ['articulation du genou', 'péroné', 'tibia']
  M24.03 (Souris intraarticulaire - " Avant-bras ") : ['articulation du poignet', 'cubitus', 'radius']
  M62.38 (Syndrome d'immobilité (paraplégique) - " Autres ") : ['colonne vertébrale', 'cou', 'crâne']
  M14.67 (Arthropathie nerveuse - " Cheville et pied ") : ['articulation de la cheville', 'autres articulations du pied', 'métatarse']
  M17.01 (Gonarthrose prim

## Question 4 — Cas particulier des codes post-2006

Un code est post-2006 si toute son existence dans le CSV vient d'ANS. Heuristique simple : aucune ligne avec `source=CIM-10` et `source_level=code`.

Code témoin attendu : U07.1 (COVID-19, ajouté en 2020).

In [9]:
print("=" * 70)
print("QUESTION 4 — Cas particulier des codes post-2006")
print("=" * 70)

codes_with_ofs_any = df.filter(
    (pl.col("source") == "CIM-10") & (pl.col("source_level") == "code")
).select("code").unique()
all_codes = df.select("code").unique()
codes_post_2006 = all_codes.join(codes_with_ofs_any, on="code", how="anti")
print(f"\nNombre de codes post-2006 estimés : {codes_post_2006.height}")

# Combien ont des synonymes ANS ?
codes_post_with_syn = codes_post_2006.join(
    df.filter((pl.col("type") == "synonyme") & (pl.col("source") == "ANS"))
      .select("code")
      .unique(),
    on="code",
    how="inner",
)
print(f"Dont avec synonymes ANS : {codes_post_with_syn.height}")

QUESTION 4 — Cas particulier des codes post-2006

Nombre de codes post-2006 estimés : 12038
Dont avec synonymes ANS : 3584


## Question 5 — Distribution de longueur des synonymes ANS

Détection des cas suspects type « Tronc » (1-2 mots), pistés dans le chantier 2 (D21.6, M01.08).

In [10]:
print("=" * 70)
print("QUESTION 5 — Distribution de longueur des synonymes ANS")
print("=" * 70)

syn_ans_with_len = syn_ans_df.with_columns(
    pl.col("texte").str.len_chars().alias("nb_chars"),
    pl.col("texte").str.split(" ").list.len().alias("nb_mots"),
)

print("\nStatistiques sur nb de mots :")
print(
    syn_ans_with_len.select(
        pl.col("nb_mots").min().alias("min"),
        pl.col("nb_mots").quantile(0.25).alias("q25"),
        pl.col("nb_mots").median().alias("median"),
        pl.col("nb_mots").quantile(0.75).alias("q75"),
        pl.col("nb_mots").max().alias("max"),
        pl.col("nb_mots").mean().alias("mean"),
    )
)

print("\nNombre de synonymes ANS très courts (1-2 mots, suspects type 'Tronc') :")
courts = syn_ans_with_len.filter(pl.col("nb_mots") <= 2)
print(f"  {courts.height} synonymes ANS de 1-2 mots")

print("\n30 exemples de synonymes ANS courts :")
for row in courts.head(30).iter_rows(named=True):
    print(f"  {row['code']:10} : '{row['texte']}' ({row['nb_mots']} mot(s))")

QUESTION 5 — Distribution de longueur des synonymes ANS

Statistiques sur nb de mots :
shape: (1, 6)
┌─────┬─────┬────────┬─────┬─────┬──────────┐
│ min ┆ q25 ┆ median ┆ q75 ┆ max ┆ mean     │
│ --- ┆ --- ┆ ---    ┆ --- ┆ --- ┆ ---      │
│ u32 ┆ f64 ┆ f64    ┆ f64 ┆ u32 ┆ f64      │
╞═════╪═════╪════════╪═════╪═════╪══════════╡
│ 1   ┆ 1.0 ┆ 1.0    ┆ 3.0 ┆ 56  ┆ 2.344447 │
└─────┴─────┴────────┴─────┴─────┴──────────┘

Nombre de synonymes ANS très courts (1-2 mots, suspects type 'Tronc') :
  10833 synonymes ANS de 1-2 mots

30 exemples de synonymes ANS courts :
  A04.7      : 'Colite pseudomembraneuse' (2 mot(s))
  A06.3      : 'Amœbome SAI' (2 mot(s))
  A09.0      : 'Catarrhe intestinal' (2 mot(s))
  A09.0      : 'Diarrhée dysentérique' (2 mot(s))
  A09.0      : 'Diarrhée épidémique' (2 mot(s))
  A15.4      : 'Tuberculose ganglionnaire' (2 mot(s))
  A15.5      : 'Tuberculose de' (2 mot(s))
  A15.8      : 'Tuberculose (de)' (2 mot(s))
  A16.3      : 'Tuberculose ganglionnaire' (2 mot(

## Question 6 — Vérification ciblée sur les cas connus

M01.08 et D21.6 — cas typiques remontés dans le chantier 2 (noms anatomiques bruts comme synonymes ANS).

In [11]:
print("=" * 70)
print("QUESTION 6 — Vérification sur les cas connus")
print("=" * 70)

for code in ["M01.08", "D21.6"]:
    print(f"\n{code} :")
    desc_ofs = df.filter(
        (pl.col("code") == code)
        & (pl.col("type") == "synonyme")
        & (pl.col("source") == "CIM-10")
        & (pl.col("source_level") == "code")
    )
    incl_ofs = df.filter(
        (pl.col("code") == code)
        & (pl.col("type") == "inclusion")
        & (pl.col("source") == "CIM-10")
        & (pl.col("source_level") == "code")
    )
    syn_ans = df.filter(
        (pl.col("code") == code)
        & (pl.col("type") == "synonyme")
        & (pl.col("source") == "ANS")
    )
    print(f"  Descripteurs OFS (section 2 primaire) : {desc_ofs.height}")
    print(f"  Inclusions OFS niveau code : {incl_ofs.height}")
    print(f"  Synonymes ANS (qui activeraient fallback) : {syn_ans.height}")
    if syn_ans.height > 0:
        for row in syn_ans.iter_rows(named=True):
            print(f"    - '{row['texte']}'")

QUESTION 6 — Vérification sur les cas connus

M01.08 :
  Descripteurs OFS (section 2 primaire) : 0
  Inclusions OFS niveau code : 0
  Synonymes ANS (qui activeraient fallback) : 6
    - 'colonne vertébrale'
    - 'cou'
    - 'crâne'
    - 'côtes'
    - 'tronc'
    - 'tête'

D21.6 :
  Descripteurs OFS (section 2 primaire) : 1
  Inclusions OFS niveau code : 0
  Synonymes ANS (qui activeraient fallback) : 0


In [12]:
ans_a18 = df.filter(
    (pl.col("code") == "A18.1") & 
    (pl.col("type") == "synonyme") & 
    (pl.col("source") == "ANS")
)
print("Synonymes ANS pour A18.1 :")
for row in ans_a18.iter_rows(named=True):
    print(f"  - '{row['texte']}'")

Synonymes ANS pour A18.1 :


In [14]:
import polars as pl
from recode_icd.utils.loaders_dev import load_exploration_context
ctx = load_exploration_context()
df = ctx.flat

print("=" * 70)
print("Vérification sur A18.1 — Synonymes ANS et OFS")
print("=" * 70)

# Synonymes ANS pour A18.1
ans_a18 = df.filter(
    (pl.col("code") == "A18.1") & 
    (pl.col("type") == "synonyme") & 
    (pl.col("source") == "ANS")
)
print(f"\nSynonymes ANS pour A18.1 : {ans_a18.height}")
for row in ans_a18.iter_rows(named=True):
    print(f"  - '{row['texte']}'")

# Descripteurs OFS pour A18.1
ofs_a18 = df.filter(
    (pl.col("code") == "A18.1") & 
    (pl.col("type") == "synonyme") & 
    (pl.col("source") == "CIM-10") &
    (pl.col("source_level") == "code")
)
print(f"\nDescripteurs OFS pour A18.1 : {ofs_a18.height}")
for row in ofs_a18.iter_rows(named=True):
    print(f"  - '{row['texte']}'")

Vérification sur A18.1 — Synonymes ANS et OFS

Synonymes ANS pour A18.1 : 0

Descripteurs OFS pour A18.1 : 6
  - 'affection inflammatoire tuberculeuse des organes pelviens de la femme'
  - 'tuberculose (de) col de l'utérus'
  - 'tuberculose (de) organes génitaux de l'homme'
  - 'tuberculose (de) rénale'
  - 'tuberculose (de) uretère'
  - 'tuberculose (de) vessie'


In [17]:
df.filter(pl.col("code") == "M01.08")

code,libelle,type,source,texte,source_level,inherited_from_code,is_dagger_in_pair,is_asterisk_in_pair
str,str,str,str,str,str,str,bool,bool
"""M01.08""","""Arthrite méningococcique (A39.8) - "" Autres """"","""inclusion""","""ANS""","""colonne vertébrale côtes cou crâne tête tronc ""","""code""",null,false,false
"""M01.08""","""Arthrite méningococcique (A39.8) - "" Autres """"","""exclusion""","""ANS""","""arthrite postméningococcique (M03.0) ""","""category""","""M01.0""",false,false
"""M01.08""","""Arthrite méningococcique (A39.8) - "" Autres """"","""exclusion""","""ANS""","""arthropathie (au cours de) : - postinfectieuse et réactionnelle (M03.-) - sarcoïdose (M14.8) ""","""category""","""M01""",false,false
"""M01.08""","""Arthrite méningococcique (A39.8) - "" Autres """"","""exclusion""","""ANS""","""certaines affections dont l'origine se situe dans la période périnatale (P00-P96) certaines lésions de l’articulation temporomandibulaire (K07.6) certaines maladies infectieuses et parasitaires (A00-B99) complications de la grossesse, de l'accouchement et de la puerpéralité (O00-O99) lésions traumatiques, empoisonnements et certaines autres conséquences de causes externes (S00-T98) maladies endocriniennes, nutritionnelles et métaboliques (E00-E90) malformations congénitales et anomalies chromoso…","""chapter""","""XIII""",false,false
"""M01.08""","""Arthrite méningococcique (A39.8) - "" Autres """"","""exclusion""","""CIM-10""","""arthrite post-méningococcique""","""category""","""M01.0""",false,false
"""M01.08""","""Arthrite méningococcique (A39.8) - "" Autres """"","""exclusion""","""CIM-10""","""arthropathie (au cours de) post-infectieuse et réactionnelle""","""category""","""M01""",false,false
"""M01.08""","""Arthrite méningococcique (A39.8) - "" Autres """"","""exclusion""","""CIM-10""","""arthropathie (au cours de) sarcoïdose""","""category""","""M01""",false,false
"""M01.08""","""Arthrite méningococcique (A39.8) - "" Autres """"","""synonyme""","""ANS""","""colonne vertébrale""","""code""",null,false,false
"""M01.08""","""Arthrite méningococcique (A39.8) - "" Autres """"","""synonyme""","""ANS""","""cou""","""code""",null,false,false


In [20]:
from recode_icd.utils.loaders_dev import inspect_code

inspect_code("M01.08", ctx, verbose=True)


╔════════════════════════════════════════════════════════════════════╗
║ M01.08 — Arthrite méningococcique (A39.8)   - " Autres "           ║
╚════════════════════════════════════════════════════════════════════╝

── BLOC 1 : IDENTITÉ ─────────────────────────────────────────────────
Code         : M01.08
Libellé OFS  : arthrite méningococcique | autres
Libellé ANS  : Arthrite méningococcique (A39.8)   - " Autres "
Type         : category
Chapitre     : XIII — Maladies du système ostéoarticulaire, des muscles et du tissu conjonctif
Bloc         : M00-M25 — Arthropathies
Catégorie    : M00-M03 — Arthropathies infectieuses

── BLOC 2 : SOURCES BRUTES (avant fusion) ────────────────────────────
[OFS]
  Inclusions :
    (aucune entrée)
  Exclusions :
    (aucune entrée)
  Descripteurs / synonymes :
    (aucune entrée)
  Notes éditoriales :
    (aucune entrée)

[ANS]
  Inclusions :
    - colonne vertébrale
côtes
cou
crâne
tête
tronc

  Exclusions :
    (aucune entrée)
  Synonymes :
    - c

In [1]:
from recode_icd.utils.loaders_dev import load_exploration_context, inspect_code_extended

ctx = load_exploration_context(load_rdf=True)         # 3,5 s la 1ʳᵉ fois

In [9]:
inspect_code_extended("M01.08", ctx)


╔════════════════════════════════════════════════════════════════════╗
║ M01.08 — Arthrite méningococcique (A39.8)   - " Autres "           ║
╚════════════════════════════════════════════════════════════════════╝

── BLOC 1 : IDENTITÉ ─────────────────────────────────────────────────
Code         : M01.08
Type de nœud : leaf
Libellé OFS  : arthrite méningococcique | autres
Libellé ANS  : Arthrite méningococcique [A39.8]   - " Autres "
URI RDF      : http://data.esante.gouv.fr/atih/cim10/M01.08
Chapitre     : XIII — Maladies du système ostéoarticulaire, des muscles et du tissu conjonctif
Bloc         : M00-M25 — Arthropathies
Catégorie    : M00-M03 — Arthropathies infectieuses

── BLOC 2 : SOURCES BRUTES (avant fusion) ────────────────────────────
[OFS]
  Inclusions :
    (aucune entrée)
  Exclusions :
    (aucune entrée)
  Descripteurs / synonymes :
    (aucune entrée)
  Notes éditoriales :
    (aucune entrée)

[ANS — RDF source]
  Libellé (rdfs:label) — 1 :
    - Arthrite méningococc

## Question 7 — Tableau des 5ᵉ positions du chapitre XIII

### Contexte (pourquoi cette quantification)

Investigation menée le 2026-06-06 sur l'origine structurelle des
synonymes ANS de qualité douteuse pour les codes `M*.X8` (cas typique
M01.08 avec ses 6 « synonymes » anatomiques bruts : tête, cou,
tronc, côtes, crâne, colonne vertébrale).

**Découverte** : ces noms ne sont pas des synonymes au sens clinique
mais des **composants du tableau des 5ᵉ positions du chapitre XIII**
défini dans la CIM-10 OMS. La position `.8` (« autres ») se
décompose en 6 régions anatomiques résiduelles que l'ANS a extraites
et (mal) étiquetées en `skos:altLabel`.

Cette cellule quantifie le problème sur l'ensemble des codes `M*.X8`
pour étayer une décision politique au futur chantier 2.

Récap structurel complet :
[docs/sessions/2026-06-06_localisations_chap13_ofs.md](../../docs/sessions/2026-06-06_localisations_chap13_ofs.md)

### Sortie attendue — 4 livrables

1. **Liste canonique des 10 valeurs de la 5ᵉ position du chapitre XIII**
   (extraite empiriquement des libellés OFS `M01.0X`).
2. **Détection** : pour chaque code `M*.X8` du chapitre XIII présent
   dans le Parquet ANS, classifier ses `skos:altLabel` en
   localisation-de-5ᵉ-position vs autre.
3. **Quantification** : combien de codes touchés, combien d'altLabel
   ANS sont en réalité des extraits du tableau, ratio global.
4. **Échantillon** des codes à fort ratio (top 15) pour décision /
   curation manuelle.


In [ ]:
# Cellule des 4 livrables — chap XIII, 5ᵉ positions vs altLabel ANS.
# Voir docs/sessions/2026-06-06_localisations_chap13_ofs.md
# pour le contexte structurel complet.

from recode_icd._normalize import normalize_for_match

master = ctx.ofs["master"].filter(pl.col("valid") == 1)
libelle = ctx.ofs["libelle"].filter(pl.col("valid") == 1)
ans = ctx.ans
assert ans is not None

# === LIVRABLE 1 — Liste canonique des 5ᵉ positions chap XIII ===
# Extraction empirique des libellés systématiques (LIBELLE source='S')
# des codes M01.00..M01.09. La partie après « | » est la localisation.
ref_codes = [f"M01.0{i}" for i in range(10)]
ref_sids = master.filter(pl.col("code").is_in(ref_codes))
ref_lookup = ref_sids.select("SID", "code").join(
    libelle.filter(pl.col("source") == "S").select("SID", "libelle"),
    on="SID", how="left",
)


def _split_loc(lib: str | None) -> str | None:
    if lib is None or "|" not in lib:
        return None
    return lib.split("|", 1)[1].strip()


position5_table = (
    ref_lookup.with_columns(
        pl.col("code").str.slice(-1, 1).alias("position"),
        pl.col("libelle")
          .map_elements(_split_loc, return_dtype=pl.String)
          .alias("localisation"),
    )
    .select("position", "localisation")
    .sort("position")
)
print("=" * 70)
print("LIVRABLE 1 — Liste canonique des 5ᵉ positions chap XIII")
print("=" * 70)
print(position5_table)

# La position .8 « autres » se décompose en 6 régions anatomiques
# résiduelles (cf CIM-10 OMS chapitre XIII). Elles ne sont pas dans
# MASTER (atomisation OFS s'arrête à 5 chars), mais on les connaît
# car ce sont précisément les altLabel observés sur M01.08.
LOC_POS8_COMPOSANTS = [
    "tête", "cou", "tronc", "côtes", "crâne", "colonne vertébrale",
]
loc_norm = {
    normalize_for_match(r["localisation"]): r["localisation"]
    for r in position5_table.iter_rows(named=True)
    if r["localisation"]
}
loc_norm_pos8 = {normalize_for_match(c): c for c in LOC_POS8_COMPOSANTS}
loc_norm_all = {**loc_norm, **loc_norm_pos8}
print(f"\nLexique normalisé pour matching altLabel : {len(loc_norm_all)} entrées")

# === LIVRABLE 2 — Détection localisations dans les altLabel ANS ===
# Périmètre : tous les codes M*.X8 (5e car = 8, dc:type=category dans
# le Parquet ANS, longueur 6 chars).
chap13_x8 = ans.filter(
    pl.col("code").str.starts_with("M")
    & (pl.col("type") == "category")
    & pl.col("code").str.ends_with("8")
    & (pl.col("code").str.len_chars() == 6)
)
print("\n" + "=" * 70)
print(f"LIVRABLE 2 — Codes M*.X8 dans le Parquet ANS : {chap13_x8.height}")
print("=" * 70)

stats = []
for row in chap13_x8.iter_rows(named=True):
    syns = list(row["synonymes"] or [])
    matched = [s for s in syns if (normalize_for_match(s) or "") in loc_norm_all]
    stats.append({
        "code": row["code"],
        "label": (row["label"] or "")[:80],
        "n_altLabel_total": len(syns),
        "n_altLabel_localisation": len(matched),
        "ratio_loc": (len(matched) / len(syns)) if syns else 0.0,
    })
detection = pl.DataFrame(stats).sort("n_altLabel_localisation", descending=True)
print(detection.head(10))

# === LIVRABLE 3 — Quantification du problème ===
n_codes_with_loc = detection.filter(pl.col("n_altLabel_localisation") > 0).height
n_codes_only_loc = detection.filter(
    (pl.col("n_altLabel_total") > 0) & (pl.col("ratio_loc") == 1.0)
).height
total_altLabel = detection.select(pl.col("n_altLabel_total").sum()).item()
total_loc = detection.select(pl.col("n_altLabel_localisation").sum()).item()
pct = (100 * total_loc / total_altLabel) if total_altLabel else 0.0

print("\n" + "=" * 70)
print("LIVRABLE 3 — Quantification du problème (chap XIII, codes M*.X8)")
print("=" * 70)
print(f"  Codes M*.X8 inspectés                          : {detection.height}")
print(f"  Dont avec ≥1 altLabel-localisation             : {n_codes_with_loc}")
print(f"  Dont avec 100% des altLabel = localisations    : {n_codes_only_loc}")
print(f"  altLabel ANS totaux sur ces codes              : {total_altLabel}")
print(f"  altLabel ANS = localisations (extraits 5e pos) : {total_loc}  ({pct:.1f}%)")

# === LIVRABLE 4 — Échantillon de codes touchés ===
print("\n" + "=" * 70)
print("LIVRABLE 4 — Top 15 codes touchés (n_altLabel_loc desc, code asc)")
print("=" * 70)
top = (
    detection.filter(pl.col("n_altLabel_localisation") > 0)
    .sort(["n_altLabel_localisation", "code"], descending=[True, False])
    .head(15)
)
for r in top.iter_rows(named=True):
    print(
        f"  {r['code']:8s}  {r['n_altLabel_localisation']:2d}/{r['n_altLabel_total']:2d}  "
        f"{r['label']}"
    )
